# 01 — XGBoost Training

Dieses Notebook trainiert **XGBoost-Regressionsmodelle** auf allen 5 Datenvarianten (D1–D5).

## Was ist XGBoost?
XGBoost (eXtreme Gradient Boosting) ist ein **Gradient-Boosting-Verfahren** auf Entscheidungsbaeumen:
- Es baut sequenziell schwache Lerner (flache Baeume) auf
- Jeder neue Baum korrigiert die Fehler des bisherigen Ensembles
- Die Vorhersage ist die Summe aller Baum-Vorhersagen

## Besonderheiten in diesem Experiment
- **GPU-Training**: Wenn CUDA verfuegbar, wird `device="cuda"` gesetzt
- **Early Stopping**: Training stoppt, wenn sich der Val-MAE 20 Runden nicht verbessert
- **Histogramm-Algorithmus**: `tree_method="hist"` (schneller als exakt, GPU-kompatibel)

**Laufzeit**: ca. 2 Minuten (GPU)

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor

SCRIPT_START = time.time()

## GPU-Erkennung

XGBoost kann auf NVIDIA-GPUs trainieren. Die Erkennung laeuft ueber PyTorch (`torch.cuda.is_available()`). Auf der GPU ist das Training bei grossen Datensaetzen (D4: 14.5M Zeilen) deutlich schneller.

In [ ]:
try:
    import torch
    XGB_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    XGB_DEVICE = "cpu"

print(f"XGBoost Device: {XGB_DEVICE}")

In [ ]:
SCRIPT_DIR = Path(".").resolve().parent
DATA_DIR = SCRIPT_DIR / "data"
MODEL_DIR = SCRIPT_DIR / "models"
OUTPUT_DIR = SCRIPT_DIR / "ergebnisse"
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "travel_time"
VARIANTEN = ["D1_single", "D2_multi4", "D3_mittel6", "D4_gross10", "D5_fremd"]

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — ignoriert Nullwerte im Nenner."""
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def load_data(variant):
    """Laedt Train/Val/Test-CSVs fuer eine Datenvariante."""
    d = DATA_DIR / variant
    train = pd.read_csv(d / "train.csv")
    val = pd.read_csv(d / "val.csv")
    test = pd.read_csv(d / "test.csv")
    features = [c for c in train.columns if c != TARGET]
    return train, val, test, features

## Hyperparameter

| Parameter | Wert | Erklaerung |
|-----------|------|------------|
| n_estimators | 500 | Max. Boosting-Runden (Early Stopping begrenzt) |
| max_depth | 8 | Baumtiefe — balanciert Kapazitaet vs. Overfitting |
| learning_rate | 0.1 | Schrittweite pro Baum (kleiner = mehr Baeume noetig) |
| subsample | 0.8 | 80% der Zeilen pro Baum (Stochastic Boosting) |
| colsample_bytree | 0.8 | 80% der Features pro Baum |
| min_child_weight | 5 | Min. Summe der Instanzgewichte pro Blatt |
| reg_alpha | 0.1 | L1-Regularisierung der Blattgewichte |
| reg_lambda | 1.0 | L2-Regularisierung der Blattgewichte |
| early_stopping_rounds | 20 | Stopp wenn Val-MAE sich 20 Runden nicht bessert |

In [ ]:
results = []

for variant in VARIANTEN:
    print(f"\n{'='*60}")
    print(f"  XGBoost | {variant}")
    print(f"{'='*60}")

    t0 = time.time()
    train, val, test, features = load_data(variant)

    X_train, y_train = train[features].values, train[TARGET].values
    X_val, y_val = val[features].values, val[TARGET].values
    X_test, y_test = test[features].values, test[TARGET].values

    print(f"  Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}")
    print(f"  Features: {len(features)}")

    model = XGBRegressor(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method="hist",
        device=XGB_DEVICE,
        n_jobs=-1,
        random_state=42,
        early_stopping_rounds=20,
        eval_metric="mae",
    )

    # Training mit Val-Set fuer Early Stopping
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50,
    )

    # Metriken
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    mae_val = mean_absolute_error(y_val, y_pred_val)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = root_mean_squared_error(y_test, y_pred_test)
    mape_test = mape(y_test, y_pred_test)

    elapsed = time.time() - t0
    print(f"\n  Val  MAE: {mae_val:.2f}s")
    print(f"  Test MAE: {mae_test:.2f}s  RMSE: {rmse_test:.2f}s  MAPE: {mape_test:.1f}%")
    print(f"  Best iteration: {model.best_iteration}")
    print(f"  Dauer: {elapsed:.1f}s")

    # Modell speichern
    model_path = MODEL_DIR / f"xgboost_{variant}.joblib"
    joblib.dump(model, model_path)

    results.append({
        "experiment": f"XGB_{variant}",
        "modell": "XGBoost",
        "daten": variant,
        "train_n": len(train),
        "mae_val": round(mae_val, 2),
        "mae_test": round(mae_test, 2),
        "rmse_test": round(rmse_test, 2),
        "mape_test": round(mape_test, 1),
        "best_iter": model.best_iteration,
        "zeit_s": round(elapsed, 1),
    })

## Zusammenfassung

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "xgboost_results.csv", index=False)
results_df[["experiment", "mae_test", "rmse_test", "mape_test", "best_iter", "zeit_s"]]

In [ ]:
print(f"Gesamtlaufzeit: {time.time() - SCRIPT_START:.1f}s")